In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_68_try_3.pickle

In [ ]:
%%RecordEvent
%%time
### cell 69 ###

# Define question strings for clarity
question_of_interest_2019_cell81 = 'Have you ever used a TPU (tensor processing unit)?'
question_of_interest_2022_cell81 = 'Approximately how many times have you used a TPU (tensor processing unit)?'
# We'll work against the 2022 question text downstream
question_of_interest = question_of_interest_2022_cell81

# 1) Standardize/rename 2019 column and do GPU‐accelerated string replacements
responses_df_2019_cell10 = responses_df_2019_cell10.rename(
    columns={question_of_interest_2019_cell81: question_of_interest}
)
responses_df_2019_cell10[question_of_interest] = (
    responses_df_2019_cell10[question_of_interest]
      .str.replace('6-24 times', '6-25 times')
      .str.replace('> 25 times', 'More than 25 times')
)

# 2) Stack all years into one GPU DataFrame
combined = pd.concat(
    [
      responses_df_2019_cell10[[question_of_interest]].assign(year='2019'),
      responses_df_2020       [[question_of_interest]].assign(year='2020'),
      responses_df_2021       [[question_of_interest]].assign(year='2021'),
      responses_df_2022_cell10[[question_of_interest]].assign(year='2022')
    ],
    ignore_index=True
)

# 3) Compute counts via GPU-accelerated value_counts
gpu_counts = (
    combined
      .value_counts([question_of_interest, 'year'])
      .reset_index(name='count')
)

# 4) Compute total per year and percentage in one shot using transform
gpu_counts = (
    gpu_counts
      .assign(
        total      = gpu_counts.groupby('year')['count'].transform('sum'),
        percentage = (gpu_counts['count'] * 100 / gpu_counts['total']).round(1)
      )
      .drop('total', axis=1)
)

# 5) Final formatting
x_axis_title_cell81 = 'Frequency of TPU usage'
df_combined_cell81 = (
    gpu_counts
      .rename(columns={question_of_interest: x_axis_title_cell81})
      .sort_values(by=['year', 'percentage'], ascending=True)
)

# Show result
df_combined_cell81.info()

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_69_try_3.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_69_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[69], f)


In [ ]:
opt_output = Out.get(4)